# Capstone 2: Modeling

## Overview
This notebook builds a machine learning pipeline to predict arrival delay (`arr_delay`). The process includes cleaning the data, defining features and target, encoding categorical variables, splitting into training and test sets, scaling features, and training regression models.

Linear Regression is used as a baseline, and Ridge Regression is applied to handle high dimensionality. Model performance is evaluated using R² and RMSE to understand how well the models explain and predict arrival delays.

## Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from sklearn.linear_model import LinearRegression, Ridge 
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

## 1. Data Loading and Inspection

In [2]:
df = pd.read_csv("../data/airline_delay_cause.csv")

print(df.head())
print("Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nTop Missing Values:\n", df.isna().sum().sort_values(ascending=False).head())

   year  month carrier       carrier_name airport  \
0  2022      5      9E  Endeavor Air Inc.     ABE   
1  2022      5      9E  Endeavor Air Inc.     ABY   
2  2022      5      9E  Endeavor Air Inc.     ACK   
3  2022      5      9E  Endeavor Air Inc.     AEX   
4  2022      5      9E  Endeavor Air Inc.     AGS   

                                        airport_name  arr_flights  arr_del15  \
0  Allentown/Bethlehem/Easton, PA: Lehigh Valley ...        136.0        7.0   
1             Albany, GA: Southwest Georgia Regional         91.0       16.0   
2                  Nantucket, MA: Nantucket Memorial         19.0        2.0   
3           Alexandria, LA: Alexandria International         88.0       14.0   
4        Augusta, GA: Augusta Regional at Bush Field        181.0       19.0   

   carrier_ct  weather_ct  ...  security_ct  late_aircraft_ct  arr_cancelled  \
0        5.95        0.00  ...          0.0              1.00            0.0   
1        7.38        0.00  ...          

In [3]:
# Remove rows where target variable is missing (cannot train without target)
df_model = df.dropna(subset=['arr_delay']).copy()
df_model['arr_delay'].isna().sum()

np.int64(0)

## 2. Feature and Target Definition
- Define X (features) and y (target)
- Encode categorical variables (one-hot encoding)
- Ensure all features are numeric

In [4]:
# Define target variable
y = df_model['arr_delay']

# Define features
X = df_model.drop([
    'arr_delay',
    'arr_del15',
    'carrier_ct',
    'weather_ct',
    'nas_ct',
    'security_ct',
    'late_aircraft_ct',
    'carrier_delay',
    'weather_delay',
    'nas_delay',
    'security_delay',
    'late_aircraft_delay'
], axis=1)

X = X.fillna(0)

print(X.shape)
print(y.shape)

print(X.isna().sum().sum())
print(y.isna().sum())

(317523, 9)
(317523,)
0
0


In [5]:
# Convert categorical variables to numeric using one-hot encoding
X = pd.get_dummies(X, drop_first=True)

print("Shape:", X.shape)
print("\nDtypes:\n",X.dtypes.value_counts())

Shape: (317523, 902)

Dtypes:
 bool       897
float64      3
int64        2
Name: count, dtype: int64


## 3. Train-Test Split
- Split data into training and testing sets
- Training data used to build models
- Testing data used to evaluate performance

In [6]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(254018, 902) (63505, 902)
(254018,) (63505,)


## 4. Feature Scaling
- Apply scaling for linear models
- Fit scaler on training data only
- Transform both training and test sets

In [7]:
# Scale features for models sensitive to feature magnitude
scaler = StandardScaler()

# Fit only on training data to avoid data leakage
scaler.fit(X_train)

# Apply scaling to both training and testing data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Baseline Model: Linear Regression
- Train linear regression model
- Establish baseline performance

In [8]:
# Train baseline Linear Regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Generate predictions on test data
y_pred = model.predict(X_test_scaled)

### 5.1 Linear Regression Evaluation
- Evaluate using RMSE and R²
- Understand baseline model performance

In [9]:
# Evaluate model performance using R² and RMSE
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2:", r2)
print("RMSE:", rmse)

# Quick sanity check for RMSE relative to target scale
print("\nY data:\n", y.describe())

R2: 0.8273839963951335
RMSE: 5191.981109835182

Y data:
 count    317523.000000
mean       4209.989113
std       12519.021012
min           0.000000
25%         436.000000
50%        1201.000000
75%        3080.000000
max      433687.000000
Name: arr_delay, dtype: float64


The Linear Regression model achieved an R² of 0.827, indicating that it explains approximately 82.7% of the variance in arrival delays. The RMSE of 5,192 minutes suggests that, on average, the model’s predictions are off by about 5,192 minutes.

## 6. Regularized Model: Ridge Regression
- Train Ridge model
- Helps reduce overfitting in high dimensional data

In [10]:
# Train Ridge Regression model
ridge = Ridge(alpha=0.1)
ridge.fit(X_train_scaled, y_train)

# Generate predictions on test data
y_pred_ridge = ridge.predict(X_test_scaled)

### 6.1 Ridge Regression Evaluation
- Evaluate Ridge model
- Compare performance to baseline

In [11]:
# Evaluate model performance using R² and RMSE
r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print("Ridge R2:", r2_ridge)
print("Ridge RMSE:", rmse_ridge)

Ridge R2: 0.8273840026452789
Ridge RMSE: 5191.981015838576


Ridge Regression produced nearly identical results to Linear Regression, with no meaningful improvement in performance. This suggests the Linear model was already stable, and regularization had little impact due to strong linear relationships in the data.

## 7. Tree-Based Model: Random Forest
- Train Random Forest model
- Handles non-linear relationships and high dimensionality
- Does not require scaling

In [13]:
# Train Random Forest model
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=10,
    random_state=13,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Generate predictions on test data
y_pred_rf = rf.predict(X_test)

### 7.1 Random Forest Evaluation
- Evaluate Random Forest model
- Compare to linear models

In [15]:
# Evaluate model performance using R² and RMSE
r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("Random Forest R2:", r2_rf)
print("Random Forest RMSE:", rmse_rf)

Random Forest R2: 0.8881612473851275
Random Forest RMSE: 4179.157436798374


Random Forest performed better than both Linear and Ridge, indicating that non-linear relationships exist in the data that linear models were unable to fully capture.

Model parameters were adjusted to improve computational efficiency due to the size of the dataset.

## 8. Model Comparison
- Compare all models (RMSE and R²)
- Identify best performing model

In [16]:
# Compare performance across all models using the same metrics
print("Linear R2:", r2)
print("Linear RMSE:", rmse)

print("Ridge R2:", r2_ridge)
print("Ridge RMSE:", rmse_ridge)

print("Random Forest R2:", r2_rf)
print("Random Forest RMSE:", rmse_rf)

Linear R2: 0.8273839963951335
Linear RMSE: 5191.981109835182
Ridge R2: 0.8273840026452789
Ridge RMSE: 5191.981015838576
Random Forest R2: 0.8881612473851275
Random Forest RMSE: 4179.157436798374


## Model Comparison

Linear Regression served as a baseline with an R² of 0.827 and RMSE of ~5192. Ridge Regression produced nearly identical results, showing that regularization had little impact.

Random Forest performed better, with an R² of 0.888 and RMSE of ~4179. This suggests that non-linear relationships are present in the data.

## 9. Model Interpretation
- Explain which model performed best and why
- Connect results back to EDA findings
- Discuss impact of high dimensionality

Random Forest provided the best performance, indicating that arrival delays are influenced by more complex relationships between features. This aligns with the EDA findings, where multiple factors contributed to total delay.

Linear and Ridge models captured general trends, but were less effective at modeling these more complex patterns.

## 10. Conclusion
- Summarize results
- State best model
- Mention possible improvements

Random Forest was the best-performing model, with higher accuracy and lower error than the linear models.

This suggests that tree-based models are better suited for capturing the complexity of arrival delays. Future improvements could include tuning model parameters and creating new features.
